In [1]:
import numpy as np
import pandas as pd
from pyg_dataset import NetlistDataset

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.model_selection import GridSearchCV

In [23]:
dataset = NetlistDataset(data_dir="../data/superblue", load_pd = True, load_pe = True, pl = True, processed = False, load_indices=None)

100%|██████████| 12/12 [01:35<00:00,  7.96s/it]


In [26]:
node_features = []
node_demand = []
node_congestion = []
for i in [0, 1, 2, 3, 6, 7, 8, 9, 10, 11]:
    node_features.append(dataset[i].node_features.cpu().numpy())
    node_demand.append(dataset[i].node_demand.cpu().numpy())
    node_congestion.append(dataset[i].node_congestion.cpu().numpy())

train_features = np.concatenate(node_features)
train_demand = np.concatenate(node_demand)
train_congestion = np.concatenate(node_congestion)

test_features = dataset[5].node_features.cpu().numpy()
test_demand = dataset[5].node_demand.cpu().numpy()
test_congestion = dataset[5].node_congestion.cpu().numpy()

# idx = np.random.choice(a=[True, False], p=[0.25, 0.75], size=train_features.shape[0])
# train_features = train_features[idx]
# train_congestion = train_congestion[idx]

In [27]:
train_features.shape

(8705245, 45)

In [ ]:
model = RandomForestClassifier(n_estimators=21, n_jobs=20)
model.fit(train_features, train_congestion)

RandomForestClassifier(n_estimators=21, n_jobs=16)

In [6]:
pred = model.predict(test_features)
print('Prediction: \t', pred)
print('Actual: \t', test_congestion)
print('Confusion Matrix: \n', confusion_matrix(test_congestion, pred))
print('Precision: \t', precision_score(test_congestion, pred))
print('Recall: \t', recall_score(test_congestion, pred))
print('F score: \t', f1_score(test_congestion, pred))

Prediction: 	 [1 1 0 ... 1 0 0]
Actual: 	 [0 0 0 ... 0 0 0]
Confusion Matrix: 
 [[ 38644 143255]
 [ 64854 248481]]
Precision: 	 0.6343072885821063
Recall: 	 0.7930202498922878
F score: 	 0.704839654446148


In [22]:
parameters = {
    'n_estimators': [11, 21, 31],
    # 'max_depth': [10, 20, 30, 40, 50, 60]
}
cv = GridSearchCV(RandomForestClassifier(), param_grid=parameters, n_jobs=20, verbose=2, cv=3)
cv.fit(train_features, train_congestion)

Fitting 3 folds for each of 3 candidates, totalling 9 fits


GridSearchCV(cv=3, estimator=RandomForestClassifier(), n_jobs=20,
             param_grid={'n_estimators': [11, 21, 31]}, verbose=2)

In [25]:
cv.best_estimator_.predict(test_features)
pred = model.predict(test_features)
print('Prediction: \t', pred)
print('Actual: \t', test_congestion)
print('Confusion Matrix: \n', confusion_matrix(test_congestion, pred))
print('Precision: \t', precision_score(test_congestion, pred))
print('Recall: \t', recall_score(test_congestion, pred))
print('F score: \t', f1_score(test_congestion, pred))

Prediction: 	 [1 1 0 ... 1 0 0]
Actual: 	 [0 0 0 ... 0 0 0]
Confusion Matrix: 
 [[ 38644 143255]
 [ 64854 248481]]
Precision: 	 0.6343072885821063
Recall: 	 0.7930202498922878
F score: 	 0.704839654446148


In [ ]:
pred.mean()

0.7910119256755392

In [15]:
parameters = {
    'n_estimators': [51, 71, 91],
    'max_depth': [40, 50, 60],
    'max_features': ['sqrt', 10, 15]
}

best_model = None
best_fscore = 0
outputs = {}
for i in parameters['n_estimators']:
    for j in parameters['max_depth']:
        for k in parameters['max_features']:
            print(f'Fitting n_estimators={i}, max_depth={j}, max_features={k}')
            model = RandomForestClassifier(n_estimators=i, max_depth=j, max_features=k, n_jobs=20, class_weight='balanced', oob_score=f1_score)
            model.fit(train_features, train_congestion)
            pred = model.predict(test_features)
            print('Confusion Matrix: \n', confusion_matrix(test_congestion, pred))
            print('Precision: \t', precision_score(test_congestion, pred))
            print('Recall: \t', recall_score(test_congestion, pred))
            print('F score: \t', f1_score(test_congestion, pred))
            print()
            if f1_score(test_congestion, pred) > best_fscore:
                best_fscore = f1_score(test_congestion, pred)
                best_model = model
            outputs[(i, j, k)] = f1_score(test_congestion, pred)

Fitting n_estimators=51, max_depth=40, max_features=sqrt
Confusion Matrix: 
 [[ 34627 147272]
 [ 66543 246792]]
Precision: 	 0.6262739047464372
Recall: 	 0.7876298530326966
F score: 	 0.6977448370721474

Fitting n_estimators=51, max_depth=40, max_features=10
Confusion Matrix: 
 [[  4749 177150]
 [  5318 308017]]
Precision: 	 0.6348679939072526
Recall: 	 0.9830277498523944
F score: 	 0.7714871096127498

Fitting n_estimators=51, max_depth=40, max_features=15
Confusion Matrix: 
 [[ 19463 162436]
 [ 30004 283331]]
Precision: 	 0.635603353321354
Recall: 	 0.9042430625369015
F score: 	 0.7464899315243538

Fitting n_estimators=51, max_depth=50, max_features=sqrt
Confusion Matrix: 
 [[ 19508 162391]
 [ 43219 270116]]
Precision: 	 0.6245355566499502
Recall: 	 0.8620677549587502
F score: 	 0.7243249910839026

Fitting n_estimators=51, max_depth=50, max_features=10
Confusion Matrix: 
 [[  7048 174851]
 [ 11291 302044]]
Precision: 	 0.6333553507585528
Recall: 	 0.9639650852920995
F score: 	 0.76444

In [16]:
best_fscore

0.7742454978780601

In [17]:
outputs

{(51, 40, 'sqrt'): 0.6977448370721474,
 (51, 40, 10): 0.7714871096127498,
 (51, 40, 15): 0.7464899315243538,
 (51, 50, 'sqrt'): 0.7243249910839026,
 (51, 50, 10): 0.7644457942624299,
 (51, 50, 15): 0.6629087302744802,
 (51, 60, 'sqrt'): 0.7385250442603074,
 (51, 60, 10): 0.7169964365060161,
 (51, 60, 15): 0.6823300136215644,
 (71, 40, 'sqrt'): 0.753129945546505,
 (71, 40, 10): 0.7474683708054125,
 (71, 40, 15): 0.7558653066030716,
 (71, 50, 'sqrt'): 0.7369996168996638,
 (71, 50, 10): 0.7454970572484582,
 (71, 50, 15): 0.7636450261339909,
 (71, 60, 'sqrt'): 0.7495362693027094,
 (71, 60, 10): 0.7128596222478223,
 (71, 60, 15): 0.6989562546478311,
 (91, 40, 'sqrt'): 0.7524783765370856,
 (91, 40, 10): 0.7626591108013796,
 (91, 40, 15): 0.7627317128170437,
 (91, 50, 'sqrt'): 0.7708457999791752,
 (91, 50, 10): 0.7512730323663236,
 (91, 50, 15): 0.7742454978780601,
 (91, 60, 'sqrt'): 0.7491998978757644,
 (91, 60, 10): 0.771438320000797,
 (91, 60, 15): 0.7594045463689398}

Outputs 1:
{(21, 10): 0.7632052233194013,
 (21, 20): 0.7108846906360395,
 (21, 30): 0.652303138643685,
 (21, 40): 0.7435049722826174,
 (21, 50): 0.6447141018006697,
 (21, 60): 0.6677995228989287,
 (31, 10): 0.7675760827778619,
 (31, 20): 0.7121059094204568,
 (31, 30): 0.6322721416662457,
 (31, 40): 0.5644555431977,
 (31, 50): 0.63837350257786,
 (31, 60): 0.7088235591586923,
 (41, 10): 0.7549049592257079,
 (41, 20): 0.6993801144079425,
 (41, 30): 0.7156088175409804,
 (41, 40): 0.6385321661082405,
 (41, 50): 0.6887231901109961,
 (41, 60): 0.6961426195764673,
 (51, 10): 0.7514667615709758,
 (51, 20): 0.5680016478026177,
 (51, 30): 0.7058983548996843,
 (51, 40): 0.6557705526804408,
 (51, 50): 0.7514077314866674,
 (51, 60): 0.7345405558067473}

Outputs 2:
{(31, 10): 0.5627610226962656,
 (31, 20): 0.6187010277888448,
 (31, 30): 0.7595434031025831,
 (31, 40): 0.7511972663441229,
 (31, 50): 0.7391442933731855,
 (31, 60): 0.7125275386910057,
 (51, 10): 0.43700242573666837,
 (51, 20): 0.6066172104475603,
 (51, 30): 0.7530491131978403,
 (51, 40): 0.7340192702208111,
 (51, 50): 0.7644362000812971,
 (51, 60): 0.7692792108587528,
 (71, 10): 0.6720527382814333,
 (71, 20): 0.7635207408011084,
 (71, 30): 0.7490136025189272,
 (71, 40): 0.6972824782383485,
 (71, 50): 0.7628335742146467,
 (71, 60): 0.7652448082001804,
 (91, 10): 0.603177512985029,
 (91, 20): 0.7340217568133389,
 (91, 30): 0.7192875838300418,
 (91, 40): 0.7558026878879253,
 (91, 50): 0.7052226199076502,
 (91, 60): 0.7481652190253476}

Outputs 3:
{(51, 40, 'sqrt'): 0.6977448370721474,
 (51, 40, 10): 0.7714871096127498,
 (51, 40, 15): 0.7464899315243538,
 (51, 50, 'sqrt'): 0.7243249910839026,
 (51, 50, 10): 0.7644457942624299,
 (51, 50, 15): 0.6629087302744802,
 (51, 60, 'sqrt'): 0.7385250442603074,
 (51, 60, 10): 0.7169964365060161,
 (51, 60, 15): 0.6823300136215644,
 (71, 40, 'sqrt'): 0.753129945546505,
 (71, 40, 10): 0.7474683708054125,
 (71, 40, 15): 0.7558653066030716,
 (71, 50, 'sqrt'): 0.7369996168996638,
 (71, 50, 10): 0.7454970572484582,
 (71, 50, 15): 0.7636450261339909,
 (71, 60, 'sqrt'): 0.7495362693027094,
 (71, 60, 10): 0.7128596222478223,
 (71, 60, 15): 0.6989562546478311,
 (91, 40, 'sqrt'): 0.7524783765370856,
 (91, 40, 10): 0.7626591108013796,
 (91, 40, 15): 0.7627317128170437,
 (91, 50, 'sqrt'): 0.7708457999791752,
 (91, 50, 10): 0.7512730323663236,
 (91, 50, 15): 0.7742454978780601,
 (91, 60, 'sqrt'): 0.7491998978757644,
 (91, 60, 10): 0.771438320000797,
 (91, 60, 15): 0.7594045463689398}

No PD:
{(51, 10, 'sqrt'): 0.01786915533658629,
 (51, 20, 'sqrt'): 0.6183044860591485,
 (51, 30, 'sqrt'): 0.3934188078258706,
 (51, 40, 'sqrt'): 0.30675573856246796,
 (51, 50, 'sqrt'): 0.492450161490138,
 (51, 60, 'sqrt'): 0.6837954869901496,
 (71, 10, 'sqrt'): 0.08679597042259007,
 (71, 20, 'sqrt'): 0.27157394827233344,
 (71, 30, 'sqrt'): 0.6247494810969886,
 (71, 40, 'sqrt'): 0.7477288248500191,
 (71, 50, 'sqrt'): 0.32104395327459456,
 (71, 60, 'sqrt'): 0.7388953723771644,
 (91, 10, 'sqrt'): 0.0044690888024497225,
 (91, 20, 'sqrt'): 0.034510308152272295,
 (91, 30, 'sqrt'): 0.7223197958098716,
 (91, 40, 'sqrt'): 0.7294011480047529,
 (91, 50, 'sqrt'): 0.6630804059663451,
 (91, 60, 'sqrt'): 0.7247615664198069}

No eigenvectors:
{(51, 30, 'sqrt'): 0.7009650131883198,
 (51, 30, 10): 0.7000023279770461,
 (51, 30, 15): 0.6978816428504716,
 (51, 45, 'sqrt'): 0.7129942926450501,
 (51, 45, 10): 0.7143805272084726,
 (51, 45, 15): 0.7195005060721601,
 (51, 60, 'sqrt'): 0.7092920631124624,
 (51, 60, 10): 0.7192473268188404,
 (51, 60, 15): 0.7147201031176624,
 (71, 30, 'sqrt'): 0.7003699672298196,
 (71, 30, 10): 0.7071719710583222,
 (71, 30, 15): 0.7081674539639771,
 (71, 45, 'sqrt'): 0.7203049855019553,
 (71, 45, 10): 0.7211442653411211,
 (71, 45, 15): 0.7233359388923896,
 (71, 60, 'sqrt'): 0.7122498501080353,
 (71, 60, 10): 0.7111488329453306,
 (71, 60, 15): 0.7234105168158905,
 (91, 30, 'sqrt'): 0.7019456129482254,
 (91, 30, 10): 0.7078305194935951,
 (91, 30, 15): 0.7078846601975265,
 (91, 45, 'sqrt'): 0.7184684970036362,
 (91, 45, 10): 0.7157802964254577,
 (91, 45, 15): 0.7240297182635922,
 (91, 60, 'sqrt'): 0.7220081378482678,
 (91, 60, 10): 0.7235755770079991,
 (91, 60, 15): 0.7195111346735895}

In [13]:
parameters = {
    'n_estimators': [51, 71, 91],
    'max_depth': [10, 20, 30, 40, 50, 60],
    'max_features': ['sqrt']
}

best_model = None
best_fscore = 0
outputs = {}
for i in parameters['n_estimators']:
    for j in parameters['max_depth']:
        for k in parameters['max_features']:
            print(f'Fitting n_estimators={i}, max_depth={j}, max_features={k}')
            model = RandomForestClassifier(n_estimators=i, max_depth=j, max_features=k, n_jobs=20, class_weight='balanced', oob_score=f1_score)
            model.fit(train_features, train_congestion)
            pred = model.predict(test_features)
            print('Confusion Matrix: \n', confusion_matrix(test_congestion, pred))
            print('Precision: \t', precision_score(test_congestion, pred))
            print('Recall: \t', recall_score(test_congestion, pred))
            print('F score: \t', f1_score(test_congestion, pred))
            print()
            if f1_score(test_congestion, pred) > best_fscore:
                best_fscore = f1_score(test_congestion, pred)
                best_model = model
            outputs[(i, j, k)] = f1_score(test_congestion, pred)

Fitting n_estimators=51, max_depth=10, max_features=sqrt
Confusion Matrix: 
 [[181428    471]
 [310506   2829]]
Precision: 	 0.8572727272727273
Recall: 	 0.00902867537938628
F score: 	 0.01786915533658629

Fitting n_estimators=51, max_depth=20, max_features=sqrt
Confusion Matrix: 
 [[ 82163  99736]
 [128487 184848]]
Precision: 	 0.64953757062941
Recall: 	 0.5899372875676193
F score: 	 0.6183044860591485

Fitting n_estimators=51, max_depth=30, max_features=sqrt
Confusion Matrix: 
 [[136919  44980]
 [225591  87744]]
Precision: 	 0.6611012326331335
Recall: 	 0.280032553018335
F score: 	 0.3934188078258706

Fitting n_estimators=51, max_depth=40, max_features=sqrt
Confusion Matrix: 
 [[142411  39488]
 [249416  63919]]
Precision: 	 0.6181303006566287
Recall: 	 0.20399572342700306
F score: 	 0.30675573856246796

Fitting n_estimators=51, max_depth=50, max_features=sqrt
Confusion Matrix: 
 [[117390  64509]
 [189910 123425]]
Precision: 	 0.6567465173944044
Recall: 	 0.3939074792155361
F score: 	

In [14]:
outputs

{(51, 10, 'sqrt'): 0.01786915533658629,
 (51, 20, 'sqrt'): 0.6183044860591485,
 (51, 30, 'sqrt'): 0.3934188078258706,
 (51, 40, 'sqrt'): 0.30675573856246796,
 (51, 50, 'sqrt'): 0.492450161490138,
 (51, 60, 'sqrt'): 0.6837954869901496,
 (71, 10, 'sqrt'): 0.08679597042259007,
 (71, 20, 'sqrt'): 0.27157394827233344,
 (71, 30, 'sqrt'): 0.6247494810969886,
 (71, 40, 'sqrt'): 0.7477288248500191,
 (71, 50, 'sqrt'): 0.32104395327459456,
 (71, 60, 'sqrt'): 0.7388953723771644,
 (91, 10, 'sqrt'): 0.0044690888024497225,
 (91, 20, 'sqrt'): 0.034510308152272295,
 (91, 30, 'sqrt'): 0.7223197958098716,
 (91, 40, 'sqrt'): 0.7294011480047529,
 (91, 50, 'sqrt'): 0.6630804059663451,
 (91, 60, 'sqrt'): 0.7247615664198069}

In [15]:
best_fscore

0.7477288248500191

In [19]:
parameters = {
    'n_estimators': [51, 71, 91],
    'max_depth': [30, 45, 60],
    'max_features': ['sqrt', 10, 15]
}

best_model = None
best_fscore = 0
outputs = {}
for i in parameters['n_estimators']:
    for j in parameters['max_depth']:
        for k in parameters['max_features']:
            print(f'Fitting n_estimators={i}, max_depth={j}, max_features={k}')
            model = RandomForestClassifier(n_estimators=i, max_depth=j, max_features=k, n_jobs=20, class_weight='balanced', oob_score=f1_score)
            model.fit(train_features, train_congestion)
            pred = model.predict(test_features)
            print('Confusion Matrix: \n', confusion_matrix(test_congestion, pred))
            print('Precision: \t', precision_score(test_congestion, pred))
            print('Recall: \t', recall_score(test_congestion, pred))
            print('F score: \t', f1_score(test_congestion, pred))
            print()
            if f1_score(test_congestion, pred) > best_fscore:
                best_fscore = f1_score(test_congestion, pred)
                best_model = model
            outputs[(i, j, k)] = f1_score(test_congestion, pred)

Fitting n_estimators=51, max_depth=30, max_features=sqrt
Confusion Matrix: 
 [[ 48049 133850]
 [ 72032 241303]]
Precision: 	 0.6432122360743483
Recall: 	 0.7701118611071218
F score: 	 0.7009650131883198

Fitting n_estimators=51, max_depth=30, max_features=10
Confusion Matrix: 
 [[ 48495 133404]
 [ 72782 240553]]
Precision: 	 0.6432637977093623
Recall: 	 0.7677182568177828
F score: 	 0.7000023279770461

Fitting n_estimators=51, max_depth=30, max_features=15
Confusion Matrix: 
 [[ 49097 132802]
 [ 74224 239111]]
Precision: 	 0.6429218661353596
Recall: 	 0.7631161536374806
F score: 	 0.6978816428504716

Fitting n_estimators=51, max_depth=45, max_features=sqrt
Confusion Matrix: 
 [[ 41078 140821]
 [ 61735 251600]]
Precision: 	 0.6411481546604285
Recall: 	 0.8029744522635518
F score: 	 0.7129942926450501

Fitting n_estimators=51, max_depth=45, max_features=10
Confusion Matrix: 
 [[ 41958 139941]
 [ 61463 251872]]
Precision: 	 0.642837271861832
Recall: 	 0.8038425327524854
F score: 	 0.71438

In [20]:
outputs

{(51, 30, 'sqrt'): 0.7009650131883198,
 (51, 30, 10): 0.7000023279770461,
 (51, 30, 15): 0.6978816428504716,
 (51, 45, 'sqrt'): 0.7129942926450501,
 (51, 45, 10): 0.7143805272084726,
 (51, 45, 15): 0.7195005060721601,
 (51, 60, 'sqrt'): 0.7092920631124624,
 (51, 60, 10): 0.7192473268188404,
 (51, 60, 15): 0.7147201031176624,
 (71, 30, 'sqrt'): 0.7003699672298196,
 (71, 30, 10): 0.7071719710583222,
 (71, 30, 15): 0.7081674539639771,
 (71, 45, 'sqrt'): 0.7203049855019553,
 (71, 45, 10): 0.7211442653411211,
 (71, 45, 15): 0.7233359388923896,
 (71, 60, 'sqrt'): 0.7122498501080353,
 (71, 60, 10): 0.7111488329453306,
 (71, 60, 15): 0.7234105168158905,
 (91, 30, 'sqrt'): 0.7019456129482254,
 (91, 30, 10): 0.7078305194935951,
 (91, 30, 15): 0.7078846601975265,
 (91, 45, 'sqrt'): 0.7184684970036362,
 (91, 45, 10): 0.7157802964254577,
 (91, 45, 15): 0.7240297182635922,
 (91, 60, 'sqrt'): 0.7220081378482678,
 (91, 60, 10): 0.7235755770079991,
 (91, 60, 15): 0.7195111346735895}

In [21]:
best_fscore

0.7240297182635922

In [39]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [ ]:
model = SVC()
model.fit(train_features, train_congestion)

In [ ]:
pred = model.predict(test_features)
print('Confusion Matrix: \n', confusion_matrix(test_congestion, pred))
print('Precision: \t', precision_score(test_congestion, pred))
print('Recall: \t', recall_score(test_congestion, pred))
print('F score: \t', f1_score(test_congestion, pred))

Confusion Matrix: 
 [[   655 181244]
 [   444 312891]]
Precision: 	 0.6332095479980168
Recall: 	 0.9985829862607114
F score: 	 0.7749910213382539


array([[-1.73942378e-04,  1.13141728e-04, -7.33690540e-04,
        -1.15568766e-03, -2.19003374e-02,  1.31687187e-04,
         4.43036986e-04,  1.68834578e-04, -1.15302943e-04,
         1.82923376e-04, -5.66461520e-04,  5.03092674e-05,
        -6.99801377e-04,  1.57908385e-04, -7.37463092e-04,
         5.92739743e-02, -2.59856421e-02, -2.01867907e-02,
         1.82417038e-02, -1.36448621e-02,  1.14316492e-01,
         2.44849938e-02, -4.81317015e-02,  1.95227305e-03,
         1.40438937e-04, -1.00870582e-03,  2.51604411e-04,
        -5.14384201e-04,  3.68826677e-04, -2.55219248e-04,
         8.73605467e-04, -1.27518696e-03,  2.51939346e-03,
         5.40365323e-02,  1.02190110e-02, -2.78927959e-03,
        -8.54369482e-03, -5.17024625e-03,  2.28817774e-03,
         2.94171117e-05,  1.47306618e-03,  3.99313603e-04,
         1.04941781e-04,  9.76963211e-05,  6.84679408e-05]])